In [36]:
import mlflow
mlflow.set_tracking_uri("sqlite:///C:/Dev/taxi-fare-predictor/mlflow.db")
mlflow.set_experiment("taxi-fare-prediction")

<Experiment: artifact_location='file:c:/Dev/taxi-fare-predictor/notebooks/mlruns/1', creation_time=1786302150001, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786302150001, lifecycle_stage='active', name='taxi-fare-prediction', tags={}, trace_location=None, workspace='default'>

In [37]:
import sys
sys.path.append('../src')

import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from pipeline import build_dataset

In [38]:
mlflow.set_experiment("taxi-fare-prediction")

<Experiment: artifact_location='file:c:/Dev/taxi-fare-predictor/notebooks/mlruns/1', creation_time=1786302150001, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786302150001, lifecycle_stage='active', name='taxi-fare-prediction', tags={}, trace_location=None, workspace='default'>

In [39]:
df = build_dataset()

In [40]:
TARGET = 'fare_amount'

NUMERICAL_FEATURES = ['trip_duration', 'passenger_count', 'trip_distance']
CATEGORICAL_FEATURES = ['pickup_borough', 'dropoff_borough', 'rate_category', 'pickup_hour', 'pickup_dayofweek']

In [41]:
df_sorted = df.sort_values('tpep_pickup_datetime').reset_index(drop=True)

X = pd.get_dummies(
    df_sorted[NUMERICAL_FEATURES + CATEGORICAL_FEATURES],
    columns=CATEGORICAL_FEATURES,
    drop_first=True
)

y = df_sorted[TARGET]

split_idx = int(len(df_sorted) * 0.8)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]


In [42]:
with mlflow.start_run(run_name="linear_baseline"):
    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
 
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
 
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("test_rows", len(X_test))
 
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
 
    mlflow.sklearn.log_model(model, "model")
 
    print(f"RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}")


2026/08/09 20:28:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE: 4.874 | MAE: 2.731 | R2: 0.914
